In [56]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from torch.utils.data import TensorDataset, DataLoader

In [4]:
path = Path('..//..//results//2026-05-26T10-20-50Z//stages//preprocess//nn_inputs')

acc = np.load(Path(path, 'acc_inputs.npy'))

strain = np.load(Path(path, 'strain_inputs.npy'))

temp = np.load(Path(path, 'temperature_inputs.npy'))

event_ids = np.load(Path(path, 'event_ids.npy'))

In [5]:
sensor_names = {
  "strain": [
    "NEW_S1_DO_INF_STR",
    "NEW_S1_DO_INF_STR",
    "NEW_S1_DO_INF_STR",
    "NEW_S1_DO_INF_STR",
    "NEW_S1_DO_INF_STR",
    "NEW_S1_DO_SUP_STR",
    "NEW_S1_DO_SUP_STR",
    "NEW_S1_DO_SUP_STR",
    "NEW_S1_DO_SUP_STR",
    "NEW_S1_DO_SUP_STR",
    "NEW_S1_UP_INF_STR",
    "NEW_S1_UP_INF_STR",
    "NEW_S1_UP_INF_STR",
    "NEW_S1_UP_INF_STR",
    "NEW_S1_UP_INF_STR",
    "NEW_S1_UP_SUP_STR",
    "NEW_S1_UP_SUP_STR",
    "NEW_S1_UP_SUP_STR",
    "NEW_S1_UP_SUP_STR",
    "NEW_S1_UP_SUP_STR",
    "NEW_S2_DO_INF_STR",
    "NEW_S2_DO_INF_STR",
    "NEW_S2_DO_INF_STR",
    "NEW_S2_DO_INF_STR",
    "NEW_S2_DO_INF_STR",
    "NEW_S2_DO_SUP_STR",
    "NEW_S2_DO_SUP_STR",
    "NEW_S2_DO_SUP_STR",
    "NEW_S2_DO_SUP_STR",
    "NEW_S2_DO_SUP_STR",
    "NEW_S2_UP_INF_STR",
    "NEW_S2_UP_INF_STR",
    "NEW_S2_UP_INF_STR",
    "NEW_S2_UP_INF_STR",
    "NEW_S2_UP_INF_STR",
    "NEW_S2_UP_SUP_STR",
    "NEW_S2_UP_SUP_STR",
    "NEW_S2_UP_SUP_STR",
    "NEW_S2_UP_SUP_STR",
    "NEW_S2_UP_SUP_STR"
  ],
  "acc_z": [
    "NEW_S1_DO_INT_ACC_Z",
    "NEW_S1_DO_INT_ACC_Z",
    "NEW_S1_DO_INT_ACC_Z",
    "NEW_S1_DO_INT_ACC_Z",
    "NEW_S1_DO_INT_ACC_Z",
    "NEW_S1_DO_MID_ACC_Z",
    "NEW_S1_DO_MID_ACC_Z",
    "NEW_S1_DO_MID_ACC_Z",
    "NEW_S1_DO_MID_ACC_Z",
    "NEW_S1_DO_MID_ACC_Z",
    "NEW_S1_UP_INT_ACC_Z",
    "NEW_S1_UP_INT_ACC_Z",
    "NEW_S1_UP_INT_ACC_Z",
    "NEW_S1_UP_INT_ACC_Z",
    "NEW_S1_UP_INT_ACC_Z",
    "NEW_S1_UP_MID_ACC_Z",
    "NEW_S1_UP_MID_ACC_Z",
    "NEW_S1_UP_MID_ACC_Z",
    "NEW_S1_UP_MID_ACC_Z",
    "NEW_S1_UP_MID_ACC_Z",
    "NEW_S2_DO_INT_ACC_Z",
    "NEW_S2_DO_INT_ACC_Z",
    "NEW_S2_DO_INT_ACC_Z",
    "NEW_S2_DO_INT_ACC_Z",
    "NEW_S2_DO_INT_ACC_Z",
    "NEW_S2_DO_MID_ACC_Z",
    "NEW_S2_DO_MID_ACC_Z",
    "NEW_S2_DO_MID_ACC_Z",
    "NEW_S2_DO_MID_ACC_Z",
    "NEW_S2_DO_MID_ACC_Z",
    "NEW_S2_UP_INT_ACC_Z",
    "NEW_S2_UP_INT_ACC_Z",
    "NEW_S2_UP_INT_ACC_Z",
    "NEW_S2_UP_INT_ACC_Z",
    "NEW_S2_UP_INT_ACC_Z",
    "NEW_S2_UP_MID_ACC_Z",
    "NEW_S2_UP_MID_ACC_Z",
    "NEW_S2_UP_MID_ACC_Z",
    "NEW_S2_UP_MID_ACC_Z",
    "NEW_S2_UP_MID_ACC_Z"
  ]
}


In [18]:
#Putting together input without temperature

input_merged = np.concatenate([acc[:,:,:], strain[:,:,:]], axis = 1)

In [ ]:
#Nå: lage Zhenkun sin modell og sjekke om den funker

mask = [event_id.startswith('AQUINAS_SET1') for event_id in event_ids]


array(['AQUINAS_SET1_2022_07__NEW__2022-07-01T02-39-01Z__2022-07-01T02-39-16Z',
       'AQUINAS_SET1_2022_07__NEW__2022-07-01T03-12-51Z__2022-07-01T03-13-05Z',
       'AQUINAS_SET1_2022_07__NEW__2022-07-01T03-19-57Z__2022-07-01T03-20-10Z',
       ...,
       'AQUINAS_SET5_2024_06__NEW__2024-06-30T20-50-50Z__2024-06-30T20-51-05Z',
       'AQUINAS_SET5_2024_06__NEW__2024-06-30T21-51-09Z__2024-06-30T21-51-23Z',
       'AQUINAS_SET5_2024_06__NEW__2024-06-30T23-28-51Z__2024-06-30T23-29-05Z'],
      dtype='<U69')

In [ ]:
event_masks = [[event_id.startswith(f'AQUINAS_SET{x}') for event_id in event_ids] for x in range(1,6)]

In [52]:
#Just testing out the reconstruction etc
#Splitting into training - testing - validation just by idx
#split : 70 - 20 - 10
#Could also do it randomized (according to what the others think)

set1_data = input_merged[event_masks[0],:,:]

train_data = set1_data[:int(len(set1_data)*0.7)]
test_data = set1_data[int(len(set1_data)*0.7):int(len(set1_data)*0.7) + int(len(set1_data)*0.2)]
val_data = set1_data[-int(len(set1_data)*0.1):]



In [ ]:
train_data = torch.tensor(train_data, dtype=torch.float32)
test_data = torch.tensor(test_data, dtype=torch.float32)
val_data = torch.tensor(val_data, dtype=torch.float32)


train_dataset = TensorDataset(train_data, train_data)
test_dataset = TensorDataset(test_data, test_data)
val_dataset = TensorDataset(val_data, val_data)


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

#Would maybe be better if I could specify Acc or Strain
#and then they get concatenated automatically within the object


C:\Users\lbreivik\AppData\Local\Temp\ipykernel_48660\1594281476.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_data = torch.tensor(train_data, dtype=torch.float32)
C:\Users\lbreivik\AppData\Local\Temp\ipykernel_48660\1594281476.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_data = torch.tensor(test_data, dtype=torch.float32)
C:\Users\lbreivik\AppData\Local\Temp\ipykernel_48660\1594281476.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_data = torch.tensor(val_data, dtype=torch.float32)
